# Tutorial

This script describes some introductory functions to get started for recreating the scripts for the final report of the AR-PST project of Topic 4, Stage 5.

All scripts can be found in `scripts/AR-PST Stage 5/Final Report/`.

### Data handling

It is recommended to select a specific folder where all the datasets and results should be stored in. Depending on the size of the study this may require multiple GB of space (the AR-PST final report required 400+ GB). This folder is called `base_path` throughout this tutorial (we are using an external storage at `Z://` - replace this with your defined location).

In [1]:
base_path = "Z://"

"Z://"

### 1 - Downloading ISP 2024 data
These cells will download the data from AEMO and save them in `/pisp-datasets` in the specified `base_path`. You can select specific reference traces, demand poe years, target years. Refer to the PISP documentation for details.

In [ ]:
# First we need to download the ISP AEMO data via PISP.
using PISP

# And now run the workflow to generate results for specific buildout cases
download_path = joinpath(base_path, "pisp-downloads")
output_root = joinpath(base_path, "pisp-datasets")

# Set parameters (see all parameters below)
reference_traces = [4006] #vcat([4006], collect(2011:2023))  # Use 4006 for the reference trace of the ODP
poes            = [10]    # Probability of exceedance (POE) for demand
target_years    = collect(2025:5:2040) #[2025,2026,2027,2028,2029,2030,2031,2040,2045,2050]

#ref_poe_sets = [(ref, poe) for ref in reference_traces for poe in poes]
ref_poe_sets = [(2019, 50), (2011, 10)] # Limited set for this tutorial, but can specify any combination of reference trace, POE, and target year

# Optional parameters that include buildout scenarios of storage and gas generation.
# The buildout Excel file used for the final report is stored in the repository:
buildout_filepath  = normpath(joinpath(@__DIR__, "..", "scripts", "AR-PST Stage 5", "Final Report", "DATA MANAGEMENT", "base_buildout.xlsx"))
sc_buildouts       = Dict(1 => "buildout_odp_s2", 2 => "buildout_odp_s2", 3 => "buildout_odp_s2") # Excel sheet name to use per ISP scenario
case_path_output = joinpath(output_root, "base")

In [ ]:
# This will execute the data handling - note this may take quite a while to run, especially if you are downloading the data for the first time. The datasets will be written to the output path specified above.
for (reference_trace, poe) in ref_poe_sets
    println("Creating datasets for reference trace $reference_trace and POE $poe...")
    PISP.build_ISP24_datasets(
        downloadpath       = download_path,
        download_from_AEMO = true,
        poe                = poe,
        reftrace           = reference_trace,
        years              = target_years,
        output_root        = case_path_output,
        write_csv          = true,
        write_arrow        = false,
        scenarios          = [1,2,3],
        write_traces       = true,
        buildout_filepath  = buildout_filepath,
        sc_buildouts       = sc_buildouts,)
end



### 2 - Run an adequacy study
We can run a full adequacy study including all storage operation approximations using the function `assess_adequacy()`.

In [ ]:
#%%
# Activate the repository environment (this notebook lives in tutorials/, the environment in the repository root)
using Pkg; Pkg.activate(".."); #Pkg.instantiate()
using PRAS
using Gurobi
using JuMP
using PRASNEM
using SchedNEM
using Dates
using CSV
using DataFrames
using Statistics
using StatsPlots

include("../functions/all_functions.jl")

In [ ]:
#%%

ref_poe_scen_sets = [(ref, poe, 2) for (ref, poe) in ref_poe_sets] # Use scenario 2 
target_years = collect(2025:5:2040)
samples = 500 # Reduce this to get results faster for the tutorial (e.g. 100). Use multiples of 100 to ensure full compatibility with the approach here

case_name = "baseVPP"
buildout_case =  "base" 
DER_parameters = PRASNEM.get_DER_parameters(; case="baseVPP")
genOpDetails = (uc=true, ramping=true, binary=false)
resilience_events = [""] #"heatwave-ref2017-ty2038-v5-thermal", "heatwave-ref2017-ty2038-v5-lines", "heatwave-ref2017-ty2038-v5-VRE", "heatwave-ref2017-ty2038-v5"] #"heatwave-ref2017-ty2038-v5" #"heatwave-ref2017-ty2038"

for resilience_event in resilience_events
    for ty in target_years
        for (ref, poe, scen) in ref_poe_scen_sets
            ens = assess_adequacy(ty, ref, poe, samples, scen, base_path;
                DER_parameters=DER_parameters,
                case_name=case_name,
                solver="Gurobi", # or "HiGHS" if no Gurobi licence is available
                resilience_event=resilience_event,
                case_name_buildout=buildout_case,
                genOpDetails=genOpDetails,
                )
        end
    end
end

#        # Optional keyword parameters of assess_adequacy that allow for custom configuration (with their defaults):
#        genOpDetails::NamedTuple = (uc=true, ramping=true, binary=false),
#        DER_parameters::Dict{String, Any} = PRASNEM.get_DER_parameters(),
#        add_lines::Dict{Int, Vector{String}} = PRASNEM.get_added_lines_per_year(),
#        hydro_parameters::Dict{String, Any} = PRASNEM.get_hydro_parameters(),
#        solver::String = "HiGHS", # "HiGHS" or "Gurobi",
#        sample_number_per_run::Int=100,
#        default_horizon::Int=4, min_time_after_event::Int=4, 
#        optimisation_window::Int=48, move_forward::Int=24

### 3 - Get the results
Now we can get the results with the `create_summary()` function, for example like this:

In [ ]:
r = create_summary(case_name; 
   ref_poe_scen_sets=ref_poe_scen_sets,
   target_years=target_years,
   samples=samples,
   base_path=base_path,
   storage_case="reoptimised")

# This contains three key datasets that we can use for analysis: Metrics, events, and unserved energy per sample

With this we can recreate the plot which highlights the uncertainty related to storage operation in the future NEM (see `scripts/AR-PST Stage 5/Chapter 4 - Storage/storage - risk profile.jl`).

In [ ]:
# Example for plots

r_greedy = create_summary("baseVPP"; 
   samples=500, 
   storage_case="greedy", 
   ref_poe_scen_sets=ref_poe_scen_sets)

res_greedy = unstack(r_greedy.metrics, :metric, :value)

r_derated = create_summary("baseVPP"; 
   samples=500, 
   storage_case="derated", 
   ref_poe_scen_sets=ref_poe_scen_sets)

res_derated = unstack(r_derated.metrics, :metric, :value)

r_reoptimised = create_summary("baseVPP"; 
   samples=500, 
   storage_case="reoptimised", 
   ref_poe_scen_sets=ref_poe_scen_sets)

res_reoptimised = unstack(r_reoptimised.metrics, :metric, :value)

In [ ]:

xlab = "Average annual USE [%]"
labs = ["A1: High energy"  "A2: Derated energy" "A3: Economic operation"]
lab_rel_stand = "Reliability standard"
title = "NEM-wide reliability risk"
kwargs = (legend=:topleft, c=[1 2 3], dpi=300, fillalpha=1.0, lc=:black,
   fillcolor=[1 2 3], lw=1, size=(600, 400), bottommargin=5Plots.mm, leftmargin=5Plots.mm)

res_all = hcat(res_greedy.NEUE, res_derated.NEUE, res_reoptimised.NEUE)

groupedbar(res_all ./ 1e4, label=labs, 
   legend=:outerright, c=[1 2 3]; kwargs...)
plot!([0.75,4.25], [20,20] ./ 1e4, label=lab_rel_stand, lc=:black, ls=:dash, lw=2)
xlabel!("Planning Year")
ylabel!(xlab)
title!(title)
xticks!(1:4,string.(res_greedy.year))
ylims!(0,35 ./ 1e4)
xlims!(0.5, 4.5)